# 🛡️ Complete Llama-Guard-3-1B Content Safety Benchmark on Kaggle GPU

### End-to-End Automated Pipeline:
1. **Hugging Face Authentication (`login()`)**
2. **Auto-Download & Normalize `aegis2.jsonl`** directly from HuggingFace Hub
3. **Load `meta-llama/Llama-Guard-3-1B` on GPU**
4. **High-Throughput GPU Batch Inference** (33,414 samples in minutes)
5. **Compute Full Metrics**: Accuracy, Precision, Recall, F1, FPR, FNR
6. **Per-Split & Per-Category Breakdown** (15 Safety Categories)
7. **Visual Plots & Legacy Comparison Report**

In [ ]:
# [1] Install Dependencies & Check GPU
!pip install -q transformers datasets torch accelerate bitsandbytes scikit-learn matplotlib seaborn tqdm huggingface_hub

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# [2] Hugging Face Authentication with login()
import os
from huggingface_hub import login

# Option A: Get token from Kaggle Secrets if available
token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    pass

# Option B: Or paste your token directly here:
if not token:
    token = "hf_YOUR_TOKEN_HERE"  # <-- ضع التوكن الخاص بك هنا

if token and token != "hf_YOUR_TOKEN_HERE":
    login(token=token)
    print("[✓] Logged in to Hugging Face successfully!")
else:
    print("⚠️ Please replace 'hf_YOUR_TOKEN_HERE' with your actual HuggingFace token.")


In [ ]:
# [3] Automated Dataset Downloader & Normalizer (Aegis-2.0)
import hashlib
import json
from pathlib import Path
from datasets import load_dataset

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DATASET_PATH = WORKING_DIR / "aegis2.jsonl"

def rid(source_id, prompt, ref=""):
    return f"{source_id}:" + hashlib.sha256(f"{source_id}\n{prompt}\n{ref}".encode()).hexdigest()[:20]

def label_to_expected(v):
    s = str(v).strip().lower()
    if s in {"1", "unsafe", "harmful", "jailbreak", "attack", "malicious", "true"}:
        return "RESTRICTED"
    if s in {"0", "safe", "unharmful", "benign", "false"}:
        return "NORMAL"
    return "UNKNOWN"

if not DATASET_PATH.exists() or DATASET_PATH.stat().st_size < 1000:
    print("Downloading 'nvidia/Aegis-AI-Content-Safety-Dataset-2.0' from Hugging Face Hub...")
    ds = load_dataset("nvidia/Aegis-AI-Content-Safety-Dataset-2.0", "default")
    rows = []
    src_id = "aegis2"
    src_url = "https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0"
    
    for sn, split in ds.items():
        for i, r in enumerate(split):
            prompt = r.get("prompt")
            if not prompt or not str(prompt).strip():
                continue
            lab = r.get("prompt_label")
            exp = label_to_expected(lab)
            cats = []
            vc = r.get("violated_categories")
            if vc:
                cats = [x.strip() for x in str(vc).split(",") if x.strip()]
            
            ref_id = str(r.get("id", f"{sn}:{i}"))
            rows.append({
                "id": rid(src_id, str(prompt), ref_id),
                "source_id": src_id,
                "prompt": str(prompt).strip(),
                "expected": exp,
                "raw_label": lab,
                "categories": cats,
                "language": None,
                "split": sn,
                "needs_relabel": (exp == "UNKNOWN"),
                "license": "CC-BY-4.0",
                "provenance": {"source_url": src_url, "source_ref": ref_id},
                "metadata": {}
            })
            
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATASET_PATH.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[✓] Successfully prepared {len(rows):,} samples at: {DATASET_PATH}")
else:
    print(f"[✓] Dataset already exists at: {DATASET_PATH} ({DATASET_PATH.stat().st_size / (1024*1024):.2f} MB)")


In [ ]:
# [4] Load Dataset Records into Memory
import pandas as pd

records = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Total Loaded Records: {len(records):,}")
df = pd.DataFrame(records)
print(df.groupby(['split', 'expected']).size().unstack(fill_value=0))


In [ ]:
# [5] Load Llama-Guard-3-1B on GPU
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-Guard-3-1B"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {DEVICE}...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model.eval()
print(f"[✓] Model loaded in {time.time() - t0:.2f}s!")


In [ ]:
# [6] Safety Prompt Formatter
def format_guard_prompt(prompt_text: str) -> str:
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Quick sanity check
test_formatted = format_guard_prompt("How to bypass security systems?")
print("Sample Template Output:\n", test_formatted[:250], "...")


In [ ]:
# [7] High-Throughput GPU Batch Inference Pipeline
from tqdm.auto import tqdm

results = []
t_eval_start = time.time()
total_samples = len(records)

print(f"Starting GPU Batch Evaluation on {total_samples:,} samples with Batch Size = {BATCH_SIZE}...")

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Llama Guard Inference"):
    batch = records[i:i + BATCH_SIZE]
    prompts = [format_guard_prompt(item.get("prompt", "")) for item in batch]
    
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
    
    input_lens = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_lens:]
    decoded_responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    for item, raw_resp in zip(batch, decoded_responses):
        cleaned = raw_resp.strip().lower()
        is_unsafe = "unsafe" in cleaned
        pred_class = "RESTRICTED" if is_unsafe else "NORMAL"
        
        results.append({
            "id": item.get("id"),
            "split": item.get("split", "unknown"),
            "expected": item.get("expected"),
            "predicted": pred_class,
            "categories": item.get("categories", []),
            "raw_response": raw_resp.strip(),
            "correct": (pred_class == item.get("expected")),
        })

total_time = time.time() - t_eval_start
print(f"\n[✓] Evaluation completed in {total_time:.2f}s ({total_samples/total_time:.2f} samples/s)!")


In [ ]:
# [8] Overall Metrics Calculation
def compute_metrics(res_list):
    tot = len(res_list)
    if tot == 0:
        return {}
    tp = sum(1 for r in res_list if r["expected"] == "RESTRICTED" and r["predicted"] == "RESTRICTED")
    tn = sum(1 for r in res_list if r["expected"] == "NORMAL" and r["predicted"] == "NORMAL")
    fp = sum(1 for r in res_list if r["expected"] == "NORMAL" and r["predicted"] == "RESTRICTED")
    fn = sum(1 for r in res_list if r["expected"] == "RESTRICTED" and r["predicted"] == "NORMAL")
    
    acc = (tp + tn) / tot
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    fpr = fp / (tn + fp) if (tn + fp) > 0 else 0.0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    
    return {
        "total": tot, "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": acc, "precision": prec, "recall": rec, "f1_score": f1,
        "fpr": fpr, "fnr": fnr
    }

overall = compute_metrics(results)

print("=" * 70)
print("          LLAMA-GUARD-3-1B — OVERALL EVALUATION RESULTS")
print("=" * 70)
print(f"Total Evaluated:     {overall['total']:,}")
print(f"Throughput:          {overall['total']/total_time:.2f} samples/sec")
print(f"True Positives:      {overall['tp']:,}")
print(f"True Negatives:      {overall['tn']:,}")
print(f"False Positives:     {overall['fp']:,} ({overall['fpr']*100:.2f}%)")
print(f"False Negatives:     {overall['fn']:,} ({overall['fnr']*100:.2f}%)")
print("-" * 70)
print(f"ACCURACY:            {overall['accuracy']*100:.2f}%")
print(f"PRECISION:           {overall['precision']*100:.2f}%")
print(f"RECALL (Safety):     {overall['recall']*100:.2f}%")
print(f"F1-SCORE:            {overall['f1_score']*100:.2f}%")
print("=" * 70)


In [ ]:
# [9] Performance by Split & Threat Category Breakdown
from collections import Counter

# Per Split
splits = sorted(list(set(r["split"] for r in results)))
split_data = []
for sp in splits:
    sp_res = [r for r in results if r["split"] == sp]
    m = compute_metrics(sp_res)
    split_data.append({
        "Split": sp,
        "Samples": m["total"],
        "Accuracy (%)": round(m["accuracy"] * 100, 2),
        "Precision (%)": round(m["precision"] * 100, 2),
        "Recall (%)": round(m["recall"] * 100, 2),
        "F1-Score (%)": round(m["f1_score"] * 100, 2),
        "FPR (%)": round(m["fpr"] * 100, 2),
    })
print("\n--- Performance by Split ---")
print(pd.DataFrame(split_data).to_string(index=False))

# Per Threat Category
cat_expected = Counter()
cat_detected = Counter()
for r in results:
    if r["expected"] == "RESTRICTED":
        for c in r.get("categories", []):
            cat_expected[c] += 1
            if r["predicted"] == "RESTRICTED":
                cat_detected[c] += 1

cat_rows = []
for c, total_c in cat_expected.most_common():
    det_c = cat_detected[c]
    rec_c = det_c / total_c if total_c > 0 else 0.0
    cat_rows.append({
        "Category": c,
        "Unsafe Samples": total_c,
        "Detected": det_c,
        "Recall (%)": round(rec_c * 100, 2)
    })
print("\n--- Threat Category Recall Breakdown ---")
print(pd.DataFrame(cat_rows).to_string(index=False))


In [ ]:
# [10] Visualization & Comparison with Legacy Baseline
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = [[overall['tn'], overall['fp']], [overall['fn'], overall['tp']]]
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: NORMAL', 'Pred: RESTRICTED'],
            yticklabels=['Actual: NORMAL', 'Actual: RESTRICTED'])
axes[0].set_title('Llama-Guard-3-1B — Confusion Matrix', fontsize=14, fontweight='bold')

# Category Recall Bar Chart
cat_df = pd.DataFrame(cat_rows)
sns.barplot(data=cat_df.head(10), y='Category', x='Recall (%)', palette='viridis', ax=axes[1])
axes[1].set_title('Safety Recall by Category (Top 10)', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 100)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%')

plt.tight_layout()
plt.savefig(str(WORKING_DIR / 'llama_guard_eval_charts.png'), dpi=300)
plt.show()

# Before vs After Table
comparison = pd.DataFrame([
    {"Metric": "Accuracy (%)", "Legacy Baseline (TF-IDF)": 45.32, "Llama-Guard-3-1B": round(overall['accuracy']*100, 2)},
    {"Metric": "Precision (%)", "Legacy Baseline (TF-IDF)": 58.54, "Llama-Guard-3-1B": round(overall['precision']*100, 2)},
    {"Metric": "Recall (%)", "Legacy Baseline (TF-IDF)": 23.93, "Llama-Guard-3-1B": round(overall['recall']*100, 2)},
    {"Metric": "F1-Score (%)", "Legacy Baseline (TF-IDF)": 33.98, "Llama-Guard-3-1B": round(overall['f1_score']*100, 2)},
    {"Metric": "False Positive Rate (%)", "Legacy Baseline (TF-IDF)": 24.17, "Llama-Guard-3-1B": round(overall['fpr']*100, 2)},
])

print("=" * 70)
print("           PERFORMANCE COMPARISON (BEFORE vs AFTER)")
print("=" * 70)
print(comparison.to_string(index=False))
print("=" * 70)

# Save Full JSON Report
final_report = {
    "model": MODEL_ID,
    "total_samples": overall["total"],
    "evaluation_time_seconds": round(total_time, 2),
    "throughput_samples_per_sec": round(overall["total"] / total_time, 2),
    "overall_metrics": overall,
    "split_metrics": split_data,
    "category_breakdown": cat_rows,
}

report_file = WORKING_DIR / "llama_guard_benchmark_report.json"
with open(report_file, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)
print(f"\n[✓] All reports and charts saved to {report_file.resolve()}")
